# 第九章 · 低基数效应

## 研报核心观点

传统成长因子可能**误判低基数企业**。

**案例**：某公司
- 2019-2024 年：单季净利润 2 亿+，峰值 20 亿+
- 2025 Q1：单季净利润 **-13 亿**（亏损）
- 2026 Q1：单季净利润 **-2.6 亿**（仍亏损）
- **同比"增速" = 巨大正数**（分母变负的数学游戏）

但实际上：
- 企业仍在亏损
- 不是"成长"，是"修复"
- 财报披露后股价反而 **-19%**

## 低基数定义（研报 §9.2）

> 当上市公司净利润为负，或营业收入（TTM）低于 3 亿元时，将面临重点监管风险。
>
> 参考上述规则，我们定义**低基数股票**：
> - 去年同期单季度净利润 **≤ 0**，或
> - 去年同期单季度营业收入 **< 0.75 亿元**（TTM 3 亿元 / 4）

## 策略

剔除低基数个股后，**重新计算传统成长因子**。

## 预期效果

研报指出：
- 剔除后传统成长因子 Rank IC **提升**（更纯净）
- 多空年化收益从低基数前的 16% 提升到剔除后的 16%+

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, '.')
import 因子库

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪')

## 2. 数据准备

In [ ]:
# 加载 PIT 字段
pit_profit = 因子库.load_pit_financial('net_profit_parent_company')
pit_revenue = 因子库.load_pit_financial('operating_revenue')

# 累计 -> 单季度
sq_profit, _ = 因子库.to_single_quarter(pit_profit, 'net_profit_parent_company')
sq_revenue, _ = 因子库.to_single_quarter(pit_revenue, 'operating_revenue')

print(f'净利润: {sq_profit.shape}')
print(f'营收: {sq_revenue.shape}')

# 行情数据
close_price = 因子库.load_daily('close')
returns_next_month = 因子库.next_month_return(close_price, months=1)

## 3. 低基数判定函数

In [ ]:
def is_low_base(sq_profit, sq_revenue, profit_threshold=0, revenue_threshold=0.75e8):
    # 判断每只股票在每个季度是否是低基数
    # 判定条件（任一满足即为低基数）：
    # 1. 去年同期单季度净利润 <= 0
    # 2. 去年同期单季度营业收入 < 0.75 亿元
    # Returns: DataFrame (date × stock), True 表示低基数

    # 去年同期单季度净利润
    last_year_profit = sq_profit.shift(4, axis=1)
    last_year_revenue = sq_revenue.shift(4, axis=1)

    # 低基数判断
    is_low = (last_year_profit <= profit_threshold) | (last_year_revenue < revenue_threshold)
    return is_low


# 计算低基数标记
low_base_mask = is_low_base(sq_profit, sq_revenue)
print(f'低基数矩阵: {low_base_mask.shape}')
print(f'低基数比例: {low_base_mask.sum().sum() / low_base_mask.size:.2%}')

# 查看分布
print('\n各期低基数股票数量（前10个）：')
print(low_base_mask.sum(axis=1).head(10))

## 4. 案例：低基数 vs 正常基数的对比

In [ ]:
# 找一个具体的低基数案例
sample_date = low_base_mask.index.intersection(sq_profit.columns)[-20]
print('样本日期:', sample_date)

sample_stocks = low_base_mask.loc[sample_date]
sample_stocks = sample_stocks[sample_stocks].index.tolist()[:5]
print('示例股票:', sample_stocks)

print()
print('这些股票的单季度净利润：')
for s in sample_stocks[:3]:
    print()
    print(s, ':')
    print(sq_profit.loc[s].dropna().tail(8))

## 5. 剔除低基数后的净利润同比

In [ ]:
def calc_yoy_filtered(sq_df, low_base_mask):
    # 计算净利润同比，剔除低基数个股
    # 低基数个股的同比因子置为 NaN（不参与排名）

    # 同比
    yoy = sq_df / sq_df.shift(4, axis=1) - 1

    # 剔除低基数（NaN 化）
    yoy_filtered = yoy.where(~low_base_mask)
    return yoy, yoy_filtered


yoy_raw, yoy_filtered = calc_yoy_filtered(sq_profit, low_base_mask)
print(f'原始同比: {yoy_raw.shape}')
print(f'剔除低基数后: {yoy_filtered.shape}')
print(f'\n剔除后 NaN 比例: {yoy_filtered.isna().sum().sum() / yoy_filtered.size:.2%}')

## 6. 对比：剔除前后因子表现

In [ ]:
ic_raw = 因子库.rank_ic_test(yoy_raw, returns_next_month)
stats_raw = 因子库.calc_ic_stats(ic_raw)

ic_filtered = 因子库.rank_ic_test(yoy_filtered, returns_next_month)
stats_filtered = 因子库.calc_ic_stats(ic_filtered)

print('=' * 60)
print('净利润同比 因子 - 剔除低基数前后对比')
print('=' * 60)
print(f'原始:    IC={stats_raw["mean"]:.4f}, ICIR={stats_raw["icir"]:.4f}')
print(f'剔除后:  IC={stats_filtered["mean"]:.4f}, ICIR={stats_filtered["icir"]:.4f}')
print(f'\n研报预期: 剔除后因子更纯净，IC 应提升')

# 十分组对比
decile_raw = 因子库.decile_test(yoy_raw, returns_next_month)
decile_filtered = 因子库.decile_test(yoy_filtered, returns_next_month)

if decile_raw and decile_filtered:
    ls_raw = decile_raw[max(decile_raw)] - decile_raw[min(decile_raw)]
    ls_filtered = decile_filtered[max(decile_filtered)] - decile_filtered[min(decile_filtered)]
    print(f'\n原始多空月收益:    {ls_raw:.4f}, 年化 {(1+ls_raw)**12-1:.4f}')
    print(f'剔除后多空月收益:  {ls_filtered:.4f}, 年化 {(1+ls_filtered)**12-1:.4f}')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IC 对比
ic_raw.cumsum().plot(ax=axes[0], label='原始', linewidth=1)
ic_filtered.cumsum().plot(ax=axes[0], label='剔除低基数', linewidth=1)
axes[0].set_title('累计 Rank IC 对比')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 十分组对比
if decile_raw and decile_filtered:
    groups = sorted(decile_raw.keys())
    vals_raw = [decile_raw[g] for g in groups]
    vals_filt = [decile_filtered[g] for g in groups]
    x = np.arange(len(groups))
    width = 0.35
    axes[1].bar(x - width/2, vals_raw, width, label='原始', alpha=0.7)
    axes[1].bar(x + width/2, vals_filt, width, label='剔除后', alpha=0.7)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(groups)
    axes[1].set_xlabel('十分组')
    axes[1].set_ylabel('平均月收益')
    axes[1].set_title('十分组收益对比')
    axes[1].legend()
    axes[1].axhline(0, color='black', linewidth=0.5)
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. 低基数比例的时间分布

In [ ]:
# 每年低基数占比
low_base_by_year = low_base_mask.resample('Y').mean().mean(axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(low_base_by_year.index.year, low_base_by_year.values * 100, color='darkorange', alpha=0.7)
ax.set_xlabel('年份')
ax.set_ylabel('低基数股票占比 (%)')
ax.set_title('每年低基数股票占比（应该和经济周期相关）')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('各年度低基数比例：')
for year, pct in low_base_by_year.items():
    print(f'  {year.year}: {pct:.2%}')

## 8. 因子汇总

| 因子 | 我们的 IC | 我们的 ICIR | 我们的多空年化 |
|------|----------|------------|---------------|
| 净利润同比（原始） | 见上 | 见上 | 见上 |
| 净利润同比（剔除低基数） | 见上 | 见上 | 见上 |

### 下一步
- **第十章**：综合成长补充因子（融合前几章所有因子）
- **第十一章**：策略回测